<a href="https://colab.research.google.com/github/qkhenh/ChordSensePro/blob/qkhenh/AI/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChordSense Pro — Training on Colab
GPU: L4 24GB | Runtime: GPU

**v3 dataset** (with `bass_id`) phải được upload lên Drive tại:
`MyDrive/ChordSensePro/DE/data/export/v3/`

## 0. Config

In [19]:
!git -C /content/ChordSensePro pull origin qkhenh

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 981 bytes | 981.00 KiB/s, done.
From https://github.com/qkhenh/ChordSensePro
 * branch            qkhenh     -> FETCH_HEAD
   30de9d7..7aeb7f8  qkhenh     -> origin/qkhenh
Updating 30de9d7..7aeb7f8
Fast-forward
 AI/notebooks/colab_train.ipynb | 29 ++++++++++++++++++++++-------
 1 file changed, 22 insertions(+), 7 deletions(-)


In [24]:
# Xóa v3 cũ (không có audio) rồi copy lại đầy đủ
!rm -rf /content/data_v3
DATA_SRC = "/content/drive/MyDrive/ChordSensePro/DE/data/export/v3"
DATA_DST = "/content/data_v3"
!cp -r {DATA_SRC} {DATA_DST}
# Verify
import os, pandas as pd
n = len(os.listdir(f"{DATA_DST}/train/audio"))
df = pd.read_parquet(f"{DATA_DST}/train/chords.parquet")
print("audio files:", n, "| rows:", len(df), "| bass_id:", "bass_id" in df.columns)

audio files: 4324 | rows: 149850 | bass_id: True


In [1]:
import os

# T4  16GB  → batch=16
# L4  24GB  → batch=32
# A100 40GB → batch=64
os.environ["NUM_WORKERS"] = "4"
os.environ["FP16"]        = "true"
os.environ["SEED"]        = "42"
print("Config set.")

Config set.


## 1. Mount Drive + Clone Repo

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!nvidia-smi

Tue Nov 10 15:26:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR      = "/content/ChordSensePro"

if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
!pwd

Cloning into '/content/ChordSensePro'...
remote: Enumerating objects: 1923, done.
remote: Counting objects: 100% (1923/1923), done.
remote: Compressing objects: 100% (1047/1047), done.
remote: Total 1923 (delta 946), reused 1799 (delta 833), pack-reused 0 (from 0)
Receiving objects: 100% (1923/1923), 24.55 MiB | 9.64 MiB/s, done.
Resolving deltas: 100% (946/946), done.
/content/ChordSensePro/AI


## 2. Install Dependencies

In [5]:
!pip install -q -r requirements.txt

## 3. Prepare Data v3

Copy v3 từ Drive về /content (tránh I/O chậm khi đọc trực tiếp từ Drive khi train).

In [23]:
DATA_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
DATA_DST = "/content/data_v3"

if not os.path.exists(DATA_DST):
    print("Copying v3 from Drive...")
    !cp -r {DATA_SRC} {DATA_DST}
    print("Done.")
else:
    print("Dataset already at", DATA_DST)

!ls {DATA_DST}
import pandas as pd
df = pd.read_parquet(os.path.join(DATA_DST, "train", "chords.parquet"))
print("train rows:", len(df), "| bass_id col:", "bass_id" in df.columns)

Dataset already at /content/data_v3
classes.json  config.yaml  test  train
train rows: 149850 | bass_id col: True


---
## Phase 1 — Train Head Only
Chọn `RUN_NAME`: `v3_5s_p1` hoặc `v3_3s_p1`

In [14]:
import yaml
from pathlib import Path

RUN_NAME = "v3_5s_p1"  # hoặc "v3_3s_p1"

cfg = yaml.safe_load(Path(f"configs/{RUN_NAME}.yaml").read_text())
cfg["data_dir"]       = DATA_DST
cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME)
cfg["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME, "results.csv")

os.makedirs(cfg["checkpoint_dir"], exist_ok=True)

colab_cfg = f"configs/colab_{RUN_NAME}.yaml"
Path(colab_cfg).write_text(yaml.dump(cfg, default_flow_style=False))
print("Config:", colab_cfg)
print("  data_dir      :", cfg["data_dir"])
print("  checkpoint_dir:", cfg["checkpoint_dir"])
print("  batch_size    :", cfg["train"]["batch_size"])
print("  epochs        :", cfg["train"]["epochs"])

Config: configs/colab_v3_5s_p1.yaml
  data_dir      : /content/data_v3
  checkpoint_dir: /content/drive/MyDrive/ChordSensePro/AI/results/v3_5s_p1
  batch_size    : 16
  epochs        : 20


In [30]:
!python -m src.train --config {colab_cfg}

2026-11-10 15:58:30,128 Device: cuda
2026-11-10 15:58:32,198 HTTP Request: GET https://huggingface.co/api/models/m-a-p/MERT-v1-330M "HTTP/1.1 200 OK"
W1110 15:58:34.307000 16383 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 15:58:34.346000 16383 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-11-10 15:58:34,893 HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-330M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-11-10 15:58:34,893 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster 

In [8]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST):
    !python -m src.train --config {colab_cfg} --resume {P1_LAST}
else:
    print("No checkpoint found.")

No checkpoint found.


---
## Phase 2 — LoRA Fine-tune MERT
Chạy sau khi Phase 1 xong. Cần `best.pt` từ Phase 1.

In [32]:
P2_RUN_NAME = "v3_5s_p2"  # hoặc "v3_3s_p2"
P1_RUN_NAME = "v3_5s_p1"  # phải match với Phase 1 đã chạy

cfg_p2 = yaml.safe_load(Path(f"configs/{P2_RUN_NAME}.yaml").read_text())
cfg_p2["data_dir"]       = DATA_DST
cfg_p2["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME)
cfg_p2["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME, "results.csv")

os.makedirs(cfg_p2["checkpoint_dir"], exist_ok=True)

colab_p2_cfg = f"configs/colab_{P2_RUN_NAME}.yaml"
Path(colab_p2_cfg).write_text(yaml.dump(cfg_p2, default_flow_style=False))

P1_BEST = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME, "best.pt")
print("Phase 1 best.pt:", P1_BEST)
print("Exists:", os.path.exists(P1_BEST))

Phase 1 best.pt: /content/drive/MyDrive/ChordSensePro/AI/results/v3_5s_p1/best.pt
Exists: True


In [33]:
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

2026-11-10 16:40:27,280 Device: cuda
2026-11-10 16:40:29,386 HTTP Request: GET https://huggingface.co/api/models/m-a-p/MERT-v1-330M "HTTP/1.1 200 OK"
W1110 16:40:31.825000 27451 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 16:40:31.887000 27451 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-11-10 16:40:32,502 HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-330M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-11-10 16:40:32,502 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster 

In [10]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("No Phase 2 checkpoint to resume.")

No Phase 2 checkpoint to resume.


---
## Export ONNX

In [28]:
P2_BEST  = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
OUT_ONNX = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P2_RUN_NAME}.onnx")
!python -m src.export_onnx --config {colab_p2_cfg} --checkpoint {P2_BEST} --output {OUT_ONNX} --benchmark

2026-11-10 15:57:45,955 HTTP Request: GET https://huggingface.co/api/models/m-a-p/MERT-v1-330M "HTTP/1.1 200 OK"
W1110 15:57:47.977000 16008 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 15:57:48.018000 16008 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-11-10 15:57:48,570 HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-330M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-11-10 15:57:48,570 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100% 403/

---
## Quick Inference Test

In [29]:
test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]
P2_BEST = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")

for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config {colab_p2_cfg} --checkpoint {P2_BEST} --audio {audio_path}

2026-11-10 15:57:56,839 HTTP Request: GET https://huggingface.co/api/models/m-a-p/MERT-v1-330M "HTTP/1.1 200 OK"
W1110 15:57:58.733000 16101 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 15:57:58.772000 16101 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-11-10 15:57:59,319 HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-330M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-11-10 15:57:59,320 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100% 403/